# AI-IDS — Exploratory Data Analysis

**Dataset:** CICIDS2017 (pre-cleaned CSV at `../data/cicids2017_cleaned.csv`)  
**Purpose:** Characterise the dataset before training — understand class imbalance, feature statistics, data quality, and linear separability.

**Sections**
1. Setup & load
2. Dataset overview
3. Class distribution & imbalance
4. Data quality audit
5. Feature statistics by class
6. Correlation matrix
7. Feature distributions — BENIGN vs ATTACK
8. Per-class violin plots
9. Feature importance (quick RF)
10. PCA — class separability

All figures are saved to `../docs/figures/` for the bachelor report.

---
## 1. Setup & load

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import config
from src.preprocessing.loader import DataLoader, generate_synthetic_data

FIGURES_DIR = Path('../docs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

BENIGN = config.BENIGN_LABEL
LABEL  = config.LABEL_COLUMN
FEATS  = config.SELECTED_FEATURES

In [ ]:
loader = DataLoader()
try:
    df = loader.load_cicids()
    print(f'Loaded real CICIDS2017 — {df.shape[0]:,} rows × {df.shape[1]} cols')
except FileNotFoundError:
    print('Real dataset not found — using 50 k synthetic rows for demonstration.')
    df = generate_synthetic_data(50_000)

available_feats = [f for f in FEATS if f in df.columns]
print(f'Selected features available: {len(available_feats)} / {len(FEATS)}')
df.head(3)

---
## 2. Dataset overview

In [ ]:
n_rows, n_cols = df.shape
n_numeric = df.select_dtypes(include='number').shape[1]
n_classes  = df[LABEL].nunique()

print(f'Rows      : {n_rows:>12,}')
print(f'Columns   : {n_cols:>12,}  ({n_numeric} numeric, {n_cols - n_numeric} categorical)')
print(f'Classes   : {n_classes:>12,}')
print(f'\nDtype breakdown:')
print(df.dtypes.value_counts().to_string())
print(f'\nDescriptive statistics for selected features:')
df[available_feats].describe().T[['mean', 'std', 'min', '50%', 'max']].round(2)

---
## 3. Class distribution & imbalance

The severe class imbalance is the central data challenge. It motivates:
- SMOTE oversampling for minority attack classes
- `class_weight='balanced'` in Random Forest
- Macro F1 (not accuracy) as the primary evaluation metric

In [ ]:
counts = df[LABEL].value_counts()
pct    = (counts / counts.sum() * 100).round(2)
ratio  = (counts / counts.min()).round(1)

dist_df = pd.DataFrame({
    'Count':    counts,
    'Pct (%)':  pct,
    'Ratio vs smallest': ratio,
})
print(dist_df.to_string())
print(f'\nImbalance ratio (largest / smallest): {counts.max() / counts.min():.0f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — log scale for readability
ax = axes[0]
colors = sns.color_palette('deep', n_colors=len(counts))
bars = ax.barh(counts.index, counts.values, color=colors)
ax.set_xscale('log')
ax.set_xlabel('Number of flows (log scale)')
ax.set_title('Class distribution (log scale)')
for bar, val in zip(bars, counts.values):
    ax.text(val * 1.05, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=8)

# Right — percentage pie
ax2 = axes[1]
wedge_colors = colors
ax2.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
        colors=wedge_colors, startangle=140,
        textprops={'fontsize': 8})
ax2.set_title('Class share (%)')

plt.suptitle('CICIDS2017 — Class distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png')
plt.show()

---
## 4. Data quality audit

Checks: missing values, infinite values, duplicate rows, and extreme outliers (|z| > 5σ).

In [ ]:
num_df = df.select_dtypes(include='number')

n_missing = df.isnull().sum().sum()
n_inf     = np.isinf(num_df).sum().sum()
n_dupes   = df.duplicated().sum()

print(f'Missing values : {n_missing:,}')
print(f'Infinite values: {n_inf:,}')
print(f'Duplicate rows : {n_dupes:,}  ({n_dupes/len(df)*100:.3f}%)')

In [ ]:
# Outlier rate per feature: fraction of rows with |z-score| > 5
z_scores    = (num_df - num_df.mean()) / num_df.std()
outlier_rate = (z_scores.abs() > 5).mean().sort_values(ascending=False)
top_outlier  = outlier_rate[outlier_rate > 0].head(15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_outlier.index, top_outlier.values * 100, color='salmon')
ax.set_xlabel('Rows with |z| > 5  (%)')
ax.set_title('Outlier rate per feature (top 15 by rate)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'data_quality_outliers.png')
plt.show()

print('\nTop outlier features (% rows with |z| > 5):')
print((top_outlier * 100).round(2).to_string())

In [ ]:
# Z-score distribution for the two most-skewed selected features
check_feats = [f for f in ['Flow Bytes/s', 'Flow Duration', 'Bwd Packets/s']
               if f in df.columns][:2]

fig, axes = plt.subplots(1, len(check_feats), figsize=(6 * len(check_feats), 4))
if len(check_feats) == 1:
    axes = [axes]

for ax, feat in zip(axes, check_feats):
    vals = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
    clipped = vals.clip(vals.quantile(0.01), vals.quantile(0.99))
    ax.hist(clipped, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(f'{feat}\n(1st–99th pct)')
    ax.set_ylabel('Frequency')

plt.suptitle('Feature value distributions — clipped to 1–99th percentile', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'data_quality_distributions.png')
plt.show()

---
## 5. Feature statistics by class

Median values of selected features, split by attack class. Shows which features discriminate the classes most clearly.

In [ ]:
key_feats = [
    'Flow Duration', 'Total Fwd Packets', 'Flow Bytes/s',
    'Flow Packets/s', 'Min Packet Length', 'PSH Flag Count',
    'FIN Flag Count', 'ACK Flag Count',
]
key_feats = [f for f in key_feats if f in df.columns]

medians = (df.groupby(LABEL)[key_feats]
             .median()
             .round(2))
print('Median values per class:')
medians

In [ ]:
# Normalise medians to [0,1] per feature for a readable heatmap
norm = (medians - medians.min()) / (medians.max() - medians.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(norm, annot=medians.astype(str), fmt='',
            cmap='YlOrRd', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Normalised median'})
ax.set_title('Feature medians per class (colour = normalised, text = raw median)')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_medians_by_class.png')
plt.show()

---
## 6. Correlation matrix

Pairwise Pearson correlation for the 20 selected features. Highly correlated pairs (|r| > 0.9) are redundant — if any dominate training they can inflate feature importance artificially.

In [ ]:
corr = df[available_feats].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # show lower triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.4, ax=ax,
            annot_kws={'size': 6}, cbar_kws={'shrink': 0.7})
ax.set_title('Feature correlation matrix — 20 selected features', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_heatmap.png')
plt.show()

# Highlight highly correlated pairs
high_corr = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
        .stack()
        .reset_index()
)
high_corr.columns = ['Feature A', 'Feature B', 'r']
high_corr = high_corr[high_corr['r'].abs() > 0.90].sort_values('r', key=abs, ascending=False)
print(f'Pairs with |r| > 0.90: {len(high_corr)}')
high_corr.round(3)

---
## 7. Feature distributions — BENIGN vs ATTACK

Box plots (outliers hidden) for key features, split into Normal Traffic vs all attack classes combined. This is the empirical evidence that the feature space separates benign from malicious traffic.

In [ ]:
box_feats = [
    'Flow Duration', 'Total Fwd Packets', 'Flow Bytes/s',
    'Flow Packets/s', 'Min Packet Length', 'Fwd Packet Length Mean',
    'PSH Flag Count', 'FIN Flag Count',
]
box_feats = [f for f in box_feats if f in df.columns]

df_box = df.copy()
df_box['Traffic'] = df_box[LABEL].apply(
    lambda x: 'Normal' if x == BENIGN else 'Attack'
)

n_cols_plot = 4
n_rows_plot = (len(box_feats) + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                          figsize=(5 * n_cols_plot, 4 * n_rows_plot))
axes = axes.flatten()

palette = {'Normal': '#4C72B0', 'Attack': '#DD8452'}
for ax, feat in zip(axes, box_feats):
    sns.boxplot(data=df_box, x='Traffic', y=feat, hue='Traffic',
                ax=ax, palette=palette, showfliers=False, width=0.5,
                legend=False)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')

for ax in axes[len(box_feats):]:
    ax.set_visible(False)

plt.suptitle('Feature distributions — Normal Traffic vs Attack flows\n(outliers hidden)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'boxplots_benign_vs_attack.png')
plt.show()

---
## 8. Per-class violin plots

Violin plots show the full distribution shape per class for three discriminating features. Visible differences across classes justify a multi-class (not binary) classifier.

In [ ]:
violin_feats = ['Flow Duration', 'Flow Bytes/s', 'Total Fwd Packets']
violin_feats = [f for f in violin_feats if f in df.columns]

# Cap to 5 k samples per class so rendering stays fast
df_violin = pd.concat([
    grp.sample(min(len(grp), 5_000), random_state=42)
    for _, grp in df.groupby(LABEL)
])

fig, axes = plt.subplots(len(violin_feats), 1,
                          figsize=(14, 5 * len(violin_feats)))
if len(violin_feats) == 1:
    axes = [axes]

for ax, feat in zip(axes, violin_feats):
    order = (
        df_violin.groupby(LABEL)[feat]
                 .median()
                 .sort_values()
                 .index
                 .tolist()
    )
    sns.violinplot(data=df_violin, x=LABEL, y=feat, hue=LABEL,
                   order=order, ax=ax, cut=0, inner='quartile',
                   palette='deep', legend=False)
    plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=9)
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Feature distributions per attack class (violin)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'violin_per_class.png')
plt.show()

---
## 9. Feature importance (quick Random Forest)

A lightweight Random Forest trained on 10% of the data to rank feature importance. Confirms which of the 20 selected features carry the most signal.

In [ ]:
sample = df.sample(frac=0.10, random_state=42)
X_s = sample[available_feats].replace([np.inf, -np.inf], np.nan)
X_s = X_s.fillna(X_s.median())

y_raw = sample[LABEL]
if y_raw.dtype == 'object':
    y_s = y_raw.map(config.ATTACK_LABELS)
else:
    y_s = y_raw

valid = y_s.notna()
X_s, y_s = X_s[valid], y_s[valid].astype(int)

rf_quick = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_quick.fit(X_s, y_s)

imp_df = (
    pd.DataFrame({'feature': available_feats,
                  'importance': rf_quick.feature_importances_})
      .sort_values('importance', ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(imp_df['feature'], imp_df['importance'],
               color='teal', edgecolor='none')
ax.set_xlabel('Mean decrease in impurity (normalised)')
ax.set_title('Random Forest feature importance\n(10% sample, 50 trees)', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance.png')
plt.show()

print('Top 10 features:')
imp_df.sort_values('importance', ascending=False).head(10).round(4)

---
## 10. PCA — class separability

Projects the 20 scaled features onto the first two principal components. Visible clustering confirms that a tree-based classifier can distinguish attack classes from normal traffic.

In [ ]:
# 2 k samples per class keeps PCA fast while keeping all classes visible
df_pca = pd.concat([
    grp.sample(min(len(grp), 2_000), random_state=42)
    for _, grp in df.groupby(LABEL)
])

X_pca = (
    df_pca[available_feats]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(df_pca[available_feats].median())
)
X_scaled = StandardScaler().fit_transform(X_pca)

pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X_scaled)
ev = pca.explained_variance_ratio_

labels_pca = df_pca[LABEL].values
unique_labels = sorted(set(labels_pca))
palette_pca = sns.color_palette('tab10', n_colors=len(unique_labels))

fig, ax = plt.subplots(figsize=(12, 8))
for label, color in zip(unique_labels, palette_pca):
    mask = labels_pca == label
    ax.scatter(components[mask, 0], components[mask, 1],
               label=label, alpha=0.35, s=8, color=color, rasterized=True)

ax.set_xlabel(f'PC1  ({ev[0]:.1%} variance explained)', fontsize=10)
ax.set_ylabel(f'PC2  ({ev[1]:.1%} variance explained)', fontsize=10)
ax.set_title(f'PCA — 2-component projection  |  total variance explained: {sum(ev):.1%}',
             fontsize=11)
ax.legend(loc='upper right', fontsize=8, markerscale=3, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pca_separability.png', dpi=200)
plt.show()

print(f'PC1: {ev[0]:.2%}  |  PC2: {ev[1]:.2%}  |  Total: {sum(ev):.2%}')

---
## Summary of findings

| Finding | Implication |
|---|---|
| 2.52 M flows, 7 classes | Large enough for robust training; multi-class problem |
| No missing or infinite values | Pre-cleaned CSV is ready for direct use |
| 161 duplicate rows (0.006 %) | Negligible; removed before training |
| 1 075:1 imbalance (Normal vs Bots) | SMOTE + `class_weight='balanced'` + macro F1 metric |
| Several features with |z| > 5σ | StandardScaler + ±5σ clipping in `FeatureEngineer` |
| High correlation in IAT / length groups | Expected for flow-level features; RF/XGB are robust |
| Clear PCA separation for most classes | Confirms tree-based models will achieve high accuracy |

**Figures saved to `../docs/figures/`:**
- `class_distribution.png` — Chapter 3 (dataset description)
- `data_quality_outliers.png` — Chapter 3 (data cleaning motivation)
- `data_quality_distributions.png` — Chapter 3 (skew motivation)
- `feature_medians_by_class.png` — Chapter 3 (feature engineering)
- `correlation_heatmap.png` — Chapter 3 (feature selection)
- `boxplots_benign_vs_attack.png` — Chapter 3 (ML motivation)
- `violin_per_class.png` — Chapter 3 (multi-class justification)
- `feature_importance.png` — Chapter 3 (feature selection justification)
- `pca_separability.png` — Chapter 3 (class separability evidence)

Run `python ../train_pipeline.py` next to produce the trained model, confusion matrix, and ROC curves for Chapter 4.